In [1]:
import pandas as pd 
from sklearn import RandomForestClassifier


In [32]:
df = pd.read_csv('salsabil_dataset_2000.csv')
df1 = df.copy()
df1['cwsi'] *= 100
df1.head().transpose()

,0,1,2,3,4
id,1,2,3,4,5
date,2021-10-15,2023-08-02,2022-08-02,2022-06-06,2022-10-14
region,Sfax,Nabeul,Kairouan,Béja,Gabès
latitude,34.74,36.45,35.67,36.73,33.88
longitude,10.76,10.73,10.1,9.18,9.79
soil_type,sandy-loam,sandy,clay-loam,loam,sandy
crop_type,Potato,Barley,Potato,Barley,Tomato
month,10,8,8,6,10
year,2021,2023,2022,2022,2022
lst_celsius,23.67,36.72,31.19,31.11,32.42


In [13]:
df.stress_label.unique()

array(['mild', 'medium', 'low', 'high', 'extreme'], dtype=object)

In [22]:
#!/usr/bin/env python3
"""
CWSI Stress Prediction Model - Standalone Script
Professional implementation of CWSI regression with automatic stress classification
"""

# Core data science libraries
import pandas as pd
import numpy as np

# Machine learning libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Model persistence
import joblib
import warnings
warnings.filterwarnings('ignore')

def classify_stress_level(cwsi_percentage):
    """
    Classify crop water stress level based on CWSI percentage.

    Parameters:
    cwsi_percentage (float): CWSI value as percentage (0-100)

    Returns:
    str: Stress level classification
    """
    if cwsi_percentage <= 10:
        return 'low'
    elif cwsi_percentage <= 49.7:
        
        return 'mild'
    elif cwsi_percentage < 60:
        return 'medium'
    elif cwsi_percentage <= 82:
        return 'high'
    else:  # cwsi_percentage > 82
        return 'extreme'

def main():
    print(" CWSI Stress Prediction Model")
    print("=" * 50)

    # Load dataset
    print(" Loading dataset...")
    df = pd.read_csv('salsabil_dataset_2000.csv')
    print(f"   • Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

    # Feature selection
    EXCLUDED_COLUMNS = ['id', 'date', 'year', 'stress_label', 'cwsi']
    FEATURE_COLUMNS = [col for col in df.columns if col not in EXCLUDED_COLUMNS]

    # Identify categorical columns
    categorical_columns = ['region', 'soil_type', 'crop_type']

    # Encode categorical variables
    df_processed = df.copy()
    label_encoders = {}

    print(" Encoding categorical variables...")
    for col in categorical_columns:
        encoder = LabelEncoder()
        df_processed[col] = encoder.fit_transform(df_processed[col])
        label_encoders[col] = encoder

    # Prepare feature matrix and target vector
    X = df_processed[FEATURE_COLUMNS].values
    y = df_processed['cwsi'].values

    # Train-test split (keep indices)
    X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
        X, y, np.arange(len(df)), test_size=0.2, random_state=42
    )

    # Model training
    print(" Training Random Forest Regressor...")
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    print(" Model training completed")

    # Model evaluation
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("\n Model Performance Metrics:")
    print("=" * 50)
    print(".4f")
    print(".4f")
    print(".4f")
    print(".4f")

    # Cross-validation
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    print(".4f")

    # Test stress classification
    print("\n Testing Stress Classification:")
    y_pred_percentage = y_pred * 100
    predicted_stress_levels = [classify_stress_level(pct) for pct in y_pred_percentage]
    actual_stress_levels = df.iloc[indices_test]['stress_label'].values

    correct_predictions = sum(1 for pred, actual in zip(predicted_stress_levels, actual_stress_levels) if pred == actual)
    classification_accuracy = correct_predictions / len(predicted_stress_levels) * 100

    print(".2f")

    # Save model
    print("\n Saving model...")
    joblib.dump(model, 'cwsi_regression_model.pkl')
    joblib.dump(label_encoders, 'categorical_encoders.pkl')

    model_info = {
        'features': FEATURE_COLUMNS,
        'categorical_columns': categorical_columns,
        'performance_metrics': {
            'mae': mae, 'rmse': rmse, 'r2': r2,
            'cv_r2_mean': cv_scores.mean(),
            'classification_accuracy': classification_accuracy
        }
    }
    joblib.dump(model_info, 'model_metadata.pkl')

    print(" Model saved successfully!")
    print("   • cwsi_regression_model.pkl")
    print("   • categorical_encoders.pkl")
    print("   • model_metadata.pkl")

    # Example prediction
    print("\n Example Prediction:")
    sample_data = {
        'region': 'Kairouan', 'latitude': 35.67, 'longitude': 10.10,
        'soil_type': 'clay-loam', 'crop_type': 'Wheat', 'month': 7,
        'lst_celsius': 38.5, 'ndvi': 0.32, 'savi': 0.28, 'evi': 0.25,
        'ta_celsius': 34.0, 'rh_percent': 28.0, 'wind_ms': 3.2,
        'solar_wm2': 820.0, 'vpd_kpa': 3.8, 'et0_mm_day': 7.5,
        'soil_moisture': 0.11, 'field_capacity': 0.28,
        'wilting_point': 0.10, 'irrigation_event': 0
    }

    # Prepare sample for prediction
    sample = sample_data.copy()
    for col in categorical_columns:
        sample[col] = label_encoders[col].transform([sample[col]])[0]

    feature_vector = np.array([[sample[feature] for feature in FEATURE_COLUMNS]])
    cwsi_prediction = model.predict(feature_vector)[0]
    cwsi_percentage = cwsi_prediction * 100
    stress_level = classify_stress_level(cwsi_percentage)

    print(".1f")
    print(f" Stress Level: {stress_level.upper()}")

if __name__ == "__main__":
    main()

 CWSI Stress Prediction Model
 Loading dataset...
   • Dataset shape: 2000 rows × 25 columns
 Encoding categorical variables...
 Training Random Forest Regressor...
 Model training completed

 Model Performance Metrics:
.4f
.4f
.4f
.4f
.4f

 Testing Stress Classification:
.2f

 Saving model...
 Model saved successfully!
   • cwsi_regression_model.pkl
   • categorical_encoders.pkl
   • model_metadata.pkl

 Example Prediction:
.1f
 Stress Level: MILD
